In [0]:
from pyspark.sql.functions import *
from delta.tables import DeltaTable
from datetime import datetime, timezone, UTC

import uuid

In [0]:
%sql
use catalog e_com_adb

In [0]:
%sql
create schema if not exists bronze_schema

In [0]:
%sql
drop table e_com_adb.bronze_schema.ingestion_control;

In [0]:
%sql
create table if not exists e_com_adb.bronze_schema.ingestion_control (
    layer string,
    table_name string,
    ts_col string,
    pk_col string,
    last_successful_ts timestamp,
    last_successful_pk bigint,
    last_run_id string,
    rows_written bigint,
    run_status string,
    updated_at timestamp
)
using delta

In [0]:
table_config = {
    "orders": {
        'pk_col': 'order_id',
        'ts_col': 'updated_at'
    },
    "products": {
        'pk_col': 'product_id',
        'ts_col': 'updated_at'
    },
    "payments": {
        'pk_col': 'payment_id',
        'ts_col': 'processed_at'
    }
}

bronze_run_id = str(uuid.uuid4())
print(f'current bronze id is {bronze_run_id}')

In [0]:
def get_last_successful_watermark(table_name:str):
    ctrl_row = (
        spark
        .table('e_com_adb.bronze_schema.ingestion_control')
        .where(
                (col('layer') == 'bronze') &
                (col('table_name') == table_name) &
                (col('run_status') == "success")
        )
        .orderBy(col('updated_at').desc())
        .limit(1)
        .collect()
    )

    if not ctrl_row:
        return None, None
    else:
        return ctrl_row[0]['last_successful_ts'], ctrl_row[0]['last_successful_pk']


In [0]:
d = datetime.now(UTC)
type(d)
print(d)

In [0]:
def upsert_bronze_control(table_name, ts_col, pk_col, last_ts, last_pk, rows_written, run_id):
    control_df = spark.createDataFrame(
        [(
            'bronze',
            table_name,
            ts_col,
            pk_col,
            last_ts,
            int(last_pk) if last_pk is not None else None,
            run_id,
            int(rows_written),
            "success",
            datetime.now(UTC)
    )],
        schema= """
            layer string,
            table_name string,
            ts_col string,
            pk_col string,
            last_successful_ts timestamp,
            last_successful_pk bigint,
            last_run_id string,
            rows_written bigint,
            run_status string,
            updated_at timestamp
        """
    )

    dt = DeltaTable.forName(spark, 'e_com_adb.bronze_schema.ingestion_control')

    (
        dt.alias('t')
        .merge(
            control_df.alias('s'),
            "t.table_name = s.table_name and t.layer = s.layer"
        )
        .whenMatchedUpdate(
            set = {
                "ts_col": "s.ts_col",
                "pk_col": "s.pk_col",
                "last_successful_ts": "s.last_successful_ts",
                "last_successful_pk": "s.last_successful_pk",
                "last_run_id": "s.last_run_id",
                "rows_written": "s.rows_written",
                "run_status": "s.run_status",
                "updated_at": "s.updated_at"
            }
        )
        .whenNotMatchedInsertAll()
        .execute()
    )

In [0]:
for table_name, cfg in table_config.items():
    pk_col = cfg['pk_col']
    ts_col = cfg['ts_col']

    source_table = f'e_com_azure_sql_db_catalog.dbo.{table_name}'
    target_table = f'e_com_adb.bronze_schema.{table_name}_raw'

    last_ts, last_pk = get_last_successful_watermark(table_name)

    print(f'\n == Processing {table_name} ==')
    print(f'last_ts = {last_ts}, last_pk = {last_pk}')

    source_df = (
        spark
        .read
        .table(source_table)
        .withColumn(ts_col, to_timestamp(col(ts_col)))
    )

    # rows_to_load = []

    if last_ts is None:
        rows_to_load = source_df
    else:
        rows_to_load = (
            source_df
            .where(
                (col(ts_col) > lit(last_ts)) |
                (
                    (col(ts_col) == lit(last_ts)) &
                    (col(pk_col) > lit(last_pk)
                )                
            )
        ))

    rows_to_load = (
        rows_to_load
        .withColumns(
            {
                'bronze_ingested_at': current_timestamp(),
                'bronze_run_id': lit(bronze_run_id),
                'bronze_source_table': lit(source_table),                
            }
        )
    )

    rows_count = rows_to_load.count()
    print(f"{table_name} rows_to_load: {rows_count}")

    if rows_count == 0:
        print(f"No new rows to load for {table_name}")
        upsert_bronze_control(table_name, ts_col, pk_col, last_ts, last_pk, rows_count, bronze_run_id)
        continue

    rows_to_load.write.mode('append').saveAsTable(target_table)    

    max_ts = rows_to_load.agg(max(ts_col).alias('max_ts')).collect()[0]['max_ts']
    max_pk = rows_to_load.where(col(ts_col) == max_ts).agg(max(pk_col).alias('max_pk')).collect()[0]['max_pk']

    print(f"max_ts = {max_ts}, max_pk = {max_pk}")
    upsert_bronze_control(table_name, ts_col, pk_col, max_ts, max_pk, rows_count, bronze_run_id)
    print(f'wrote {rows_count} rows to {target_table}')

In [0]:
df = (
    spark
    .read
    .table("e_com_azure_sql_db_catalog.dbo.orders")
)

In [0]:
display(df)

In [0]:
print(f'Orders Bronze : {spark.table('e_com_adb.bronze_schema.orders_raw').agg(count(col('order_id'))).collect()[0][0]}')
print(f'Products Bronze : {spark.table('e_com_adb.bronze_schema.products_raw').agg(count(col('product_id'))).collect()[0][0]}')
print(f'Payments Bronze : {spark.table("e_com_adb.bronze_schema.payments_raw").agg(count(col('payment_id'))).collect()[0][0]}')

display(spark.table('e_com_adb.bronze_schema.ingestion_control'))